In [1]:
!nvidia-smi
!nvcc --version

Sun Nov  8 17:22:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%%writefile hello.cu
#include <stdio.h>
#include <cuda_runtime.h>

__global__ void helloKernel()
{
    printf("Hello from the GPU!\n");
}

int main()
{
    printf("Hello from the CPU!\n");

    helloKernel<<<1, 1>>>();

    cudaDeviceSynchronize();

    return 0;
}

Writing hello.cu


In [3]:
!nvcc hello.cu -o hello
!./hello

Hello from the CPU!
Hello from the GPU!


In [4]:
%%writefile blocks_threads.cu
#include <stdio.h>
#include <cuda_runtime.h>

__global__ void helloKernel()
{
    printf("Block %d, Thread %d\n",
           blockIdx.x,
           threadIdx.x);
}

int main()
{
    printf("Hello from the CPU!\n");

    helloKernel<<<2, 4>>>();

    cudaDeviceSynchronize();

    return 0;
}

Writing blocks_threads.cu


In [5]:
!nvcc blocks_threads.cu -o blocks_threads
!./blocks_threads

Hello from the CPU!
Block 0, Thread 0
Block 0, Thread 1
Block 0, Thread 2
Block 0, Thread 3
Block 1, Thread 0
Block 1, Thread 1
Block 1, Thread 2
Block 1, Thread 3


In [6]:
%%writefile global_index.cu
#include <stdio.h>
#include <cuda_runtime.h>

__global__ void showIndex()
{
    int index =
        threadIdx.x +
        blockIdx.x * blockDim.x;

    printf("Block %d, Thread %d, Global index %d\n",
           blockIdx.x,
           threadIdx.x,
           index);
}

int main()
{
    showIndex<<<3, 4>>>();

    cudaDeviceSynchronize();

    return 0;
}

Writing global_index.cu


In [7]:
!nvcc global_index.cu -o global_index
!./global_index

Block 1, Thread 0, Global index 4
Block 1, Thread 1, Global index 5
Block 1, Thread 2, Global index 6
Block 1, Thread 3, Global index 7
Block 0, Thread 0, Global index 0
Block 0, Thread 1, Global index 1
Block 0, Thread 2, Global index 2
Block 0, Thread 3, Global index 3
Block 2, Thread 0, Global index 8
Block 2, Thread 1, Global index 9
Block 2, Thread 2, Global index 10
Block 2, Thread 3, Global index 11


In [8]:
%%writefile vector_add.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

__global__ void vectorAdd(const float *A,
                          const float *B,
                          float *C,
                          int N)
{
    int i =
        threadIdx.x +
        blockIdx.x * blockDim.x;

    if (i < N)
    {
        C[i] = A[i] + B[i];
    }
}

int main()
{
    const int N = 1000000;

    size_t bytes = N * sizeof(float);

    float *A = (float *)malloc(bytes);
    float *B = (float *)malloc(bytes);
    float *C = (float *)malloc(bytes);

    for (int i = 0; i < N; ++i)
    {
        A[i] = (float)i;
        B[i] = 2.0f * i;
    }

    float *dev_A;
    float *dev_B;
    float *dev_C;

    // allocate GPU memory
    cudaMalloc((void **)&dev_A, bytes);
    cudaMalloc((void **)&dev_B, bytes);
    cudaMalloc((void **)&dev_C, bytes);

    // copy A and B to GPU memory
    cudaMemcpy(dev_A, A, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(dev_B, B, bytes, cudaMemcpyHostToDevice);

    int threadsPerBlock = 256;

    int blocks =
        (N + threadsPerBlock - 1)
        / threadsPerBlock;

    // launch vectorAdd kernel
    vectorAdd<<<blocks, threadsPerBlock>>>(dev_A, dev_B, dev_C, N);

    // Wait until the GPU finishes
    cudaDeviceSynchronize();

    // copy C back to the CPU
    cudaMemcpy(C, dev_C, bytes, cudaMemcpyDeviceToHost);

    printf("C[0] = %f\n", C[0]);
    printf("C[10] = %f\n", C[10]);
    printf("C[N-1] = %f\n", C[N - 1]);

    // check every element: C[i] must be 3i
    int errors = 0;
    for (int i = 0; i < N; ++i)
    {
        if (C[i] != 3.0f * i)
        {
            errors++;
        }
    }
    printf("Errors: %d\n", errors);

    // free device memory
    cudaFree(dev_A);
    cudaFree(dev_B);
    cudaFree(dev_C);

    free(A);
    free(B);
    free(C);

    return 0;
}

Writing vector_add.cu


In [9]:
!nvcc vector_add.cu -o vector_add
!./vector_add

C[0] = 0.000000
C[10] = 30.000000
C[N-1] = 2999997.000000
Errors: 0


In [10]:
%%writefile vector_add_blocks.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

__global__ void vectorAdd(const float *A,
                          const float *B,
                          float *C,
                          int N)
{
    int i =
        threadIdx.x +
        blockIdx.x * blockDim.x;

    if (i < N)
    {
        C[i] = A[i] + B[i];
    }
}

int main(int argc, char **argv)
{
    const int N = 1000000;

    size_t bytes = N * sizeof(float);

    float *A = (float *)malloc(bytes);
    float *B = (float *)malloc(bytes);
    float *C = (float *)malloc(bytes);

    for (int i = 0; i < N; ++i)
    {
        A[i] = (float)i;
        B[i] = 2.0f * i;
    }

    float *dev_A;
    float *dev_B;
    float *dev_C;

    cudaMalloc((void **)&dev_A, bytes);
    cudaMalloc((void **)&dev_B, bytes);
    cudaMalloc((void **)&dev_C, bytes);

    cudaMemcpy(dev_A, A, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(dev_B, B, bytes, cudaMemcpyHostToDevice);

    // threads per block comes from the command line, default is 256
    int threadsPerBlock = 256;
    if (argc > 1)
    {
        threadsPerBlock = atoi(argv[1]);
    }

    int blocks =
        (N + threadsPerBlock - 1)
        / threadsPerBlock;

    vectorAdd<<<blocks, threadsPerBlock>>>(dev_A, dev_B, dev_C, N);

    cudaDeviceSynchronize();

    cudaMemcpy(C, dev_C, bytes, cudaMemcpyDeviceToHost);

    int errors = 0;
    for (int i = 0; i < N; ++i)
    {
        if (C[i] != 3.0f * i)
        {
            errors++;
        }
    }

    printf("Threads per block: %d, Blocks: %d\n", threadsPerBlock, blocks);
    printf("C[0] = %f, C[10] = %f, C[N-1] = %f, Errors: %d\n\n",
           C[0], C[10], C[N - 1], errors);

    cudaFree(dev_A);
    cudaFree(dev_B);
    cudaFree(dev_C);

    free(A);
    free(B);
    free(C);

    return 0;
}

Writing vector_add_blocks.cu


In [11]:
!nvcc vector_add_blocks.cu -o vector_add_blocks
!for t in 32 64 128 256 512; do ./vector_add_blocks $t; done

Threads per block: 32, Blocks: 31250
C[0] = 0.000000, C[10] = 30.000000, C[N-1] = 2999997.000000, Errors: 0

Threads per block: 64, Blocks: 15625
C[0] = 0.000000, C[10] = 30.000000, C[N-1] = 2999997.000000, Errors: 0

Threads per block: 128, Blocks: 7813
C[0] = 0.000000, C[10] = 30.000000, C[N-1] = 2999997.000000, Errors: 0

Threads per block: 256, Blocks: 3907
C[0] = 0.000000, C[10] = 30.000000, C[N-1] = 2999997.000000, Errors: 0

Threads per block: 512, Blocks: 1954
C[0] = 0.000000, C[10] = 30.000000, C[N-1] = 2999997.000000, Errors: 0



In [12]:
%%writefile vector_add_timing.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

__global__ void vectorAdd(const float *A,
                          const float *B,
                          float *C,
                          int N)
{
    int i =
        threadIdx.x +
        blockIdx.x * blockDim.x;

    if (i < N)
    {
        C[i] = A[i] + B[i];
    }
}

int main(int argc, char **argv)
{
    const int N = 1000000;

    size_t bytes = N * sizeof(float);

    float *A = (float *)malloc(bytes);
    float *B = (float *)malloc(bytes);
    float *C = (float *)malloc(bytes);

    for (int i = 0; i < N; ++i)
    {
        A[i] = (float)i;
        B[i] = 2.0f * i;
    }

    float *dev_A;
    float *dev_B;
    float *dev_C;

    cudaMalloc((void **)&dev_A, bytes);
    cudaMalloc((void **)&dev_B, bytes);
    cudaMalloc((void **)&dev_C, bytes);

    cudaMemcpy(dev_A, A, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(dev_B, B, bytes, cudaMemcpyHostToDevice);

    int threadsPerBlock = 256;
    if (argc > 1)
    {
        threadsPerBlock = atoi(argv[1]);
    }

    int blocks =
        (N + threadsPerBlock - 1)
        / threadsPerBlock;

    // warm up launch, not measured (the first launch is always slower)
    vectorAdd<<<blocks, threadsPerBlock>>>(dev_A, dev_B, dev_C, N);
    cudaDeviceSynchronize();

    cudaEvent_t start, stop;

    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    const int RUNS = 10;
    float total = 0.0f;

    printf("Threads per block: %d\n", threadsPerBlock);
    printf("Times (ms):");

    for (int r = 0; r < RUNS; ++r)
    {
        cudaEventRecord(start);

        vectorAdd<<<blocks, threadsPerBlock>>>(
            dev_A, dev_B, dev_C, N);

        cudaEventRecord(stop);

        cudaEventSynchronize(stop);

        float milliseconds = 0.0f;

        cudaEventElapsedTime(
            &milliseconds, start, stop);

        printf(" %.4f", milliseconds);
        total += milliseconds;
    }

    printf("\nAverage kernel time: %f ms\n", total / RUNS);

    cudaEventDestroy(start);
    cudaEventDestroy(stop);

    cudaMemcpy(C, dev_C, bytes, cudaMemcpyDeviceToHost);

    int errors = 0;
    for (int i = 0; i < N; ++i)
    {
        if (C[i] != 3.0f * i)
        {
            errors++;
        }
    }

    printf("C[0] = %f, C[10] = %f, C[N-1] = %f, Errors: %d\n\n",
           C[0], C[10], C[N - 1], errors);

    cudaFree(dev_A);
    cudaFree(dev_B);
    cudaFree(dev_C);

    free(A);
    free(B);
    free(C);

    return 0;
}

Writing vector_add_timing.cu


In [13]:
!nvcc vector_add_timing.cu -o vector_add_timing
!for t in 32 64 128 256 512; do ./vector_add_timing $t; done

Threads per block: 32
Times (ms): 0.0725 0.0685 0.0696 0.0676 0.0694 0.0693 0.0681 0.0678 0.0676 0.0676
Average kernel time: 0.068806 ms
C[0] = 0.000000, C[10] = 30.000000, C[N-1] = 2999997.000000, Errors: 0

Threads per block: 64
Times (ms): 0.0563 0.0532 0.0512 0.0512 0.0514 0.0512 0.0507 0.0512 0.0525 0.0514
Average kernel time: 0.052038 ms
C[0] = 0.000000, C[10] = 30.000000, C[N-1] = 2999997.000000, Errors: 0

Threads per block: 128
Times (ms): 0.0556 0.0532 0.0516 0.0536 0.0519 0.0514 0.0512 0.0510 0.0512 0.0510
Average kernel time: 0.052186 ms
C[0] = 0.000000, C[10] = 30.000000, C[N-1] = 2999997.000000, Errors: 0

Threads per block: 256
Times (ms): 0.0553 0.0516 0.0512 0.0510 0.0552 0.0512 0.0512 0.0512 0.0512 0.0513
Average kernel time: 0.052042 ms
C[0] = 0.000000, C[10] = 30.000000, C[N-1] = 2999997.000000, Errors: 0

Threads per block: 512
Times (ms): 0.0558 0.0528 0.0520 0.0500 0.0509 0.0512 0.0514 0.0511 0.0512 0.0507
Average kernel time: 0.051722 ms
C[0] = 0.000000, C[10] =

In [14]:
%%writefile cpu_vs_gpu.cu
#include <stdio.h>
#include <stdlib.h>
#include <chrono>
#include <cuda_runtime.h>

__global__ void vectorAdd(const float *A,
                          const float *B,
                          float *C,
                          int N)
{
    int i =
        threadIdx.x +
        blockIdx.x * blockDim.x;

    if (i < N)
    {
        C[i] = A[i] + B[i];
    }
}

// the same operation on the CPU
void cpuAdd(const float *A, const float *B, float *C, int N)
{
    for (int i = 0; i < N; ++i)
    {
        C[i] = A[i] + B[i];
    }
}

int main(int argc, char **argv)
{
    // N comes from the command line, default is 1000000
    int N = 1000000;
    if (argc > 1)
    {
        N = atoi(argv[1]);
    }

    size_t bytes = (size_t)N * sizeof(float);

    float *A = (float *)malloc(bytes);
    float *B = (float *)malloc(bytes);
    float *C_cpu = (float *)malloc(bytes);
    float *C_gpu = (float *)malloc(bytes);

    for (int i = 0; i < N; ++i)
    {
        A[i] = (float)i;
        B[i] = 2.0f * i;
    }

    const int RUNS = 10;

    // CPU: one warm up run, then the average of 10 runs
    cpuAdd(A, B, C_cpu, N);

    double cpuTotal = 0.0;
    for (int r = 0; r < RUNS; ++r)
    {
        auto t0 = std::chrono::high_resolution_clock::now();
        cpuAdd(A, B, C_cpu, N);
        auto t1 = std::chrono::high_resolution_clock::now();
        cpuTotal += std::chrono::duration<double, std::milli>(t1 - t0).count();
    }
    double cpuTime = cpuTotal / RUNS;

    // GPU
    float *dev_A;
    float *dev_B;
    float *dev_C;

    cudaMalloc((void **)&dev_A, bytes);
    cudaMalloc((void **)&dev_B, bytes);
    cudaMalloc((void **)&dev_C, bytes);

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    // time of copying A and B from host to device
    cudaEventRecord(start);
    cudaMemcpy(dev_A, A, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(dev_B, B, bytes, cudaMemcpyHostToDevice);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    float h2dTime = 0.0f;
    cudaEventElapsedTime(&h2dTime, start, stop);

    int threadsPerBlock = 256;
    int blocks =
        (N + threadsPerBlock - 1)
        / threadsPerBlock;

    // kernel: one warm up launch, then the average of 10 launches
    vectorAdd<<<blocks, threadsPerBlock>>>(dev_A, dev_B, dev_C, N);
    cudaDeviceSynchronize();

    float kernelTotal = 0.0f;
    for (int r = 0; r < RUNS; ++r)
    {
        cudaEventRecord(start);
        vectorAdd<<<blocks, threadsPerBlock>>>(dev_A, dev_B, dev_C, N);
        cudaEventRecord(stop);
        cudaEventSynchronize(stop);

        float milliseconds = 0.0f;
        cudaEventElapsedTime(&milliseconds, start, stop);
        kernelTotal += milliseconds;
    }
    float kernelTime = kernelTotal / RUNS;

    // time of copying C from device to host
    cudaEventRecord(start);
    cudaMemcpy(C_gpu, dev_C, bytes, cudaMemcpyDeviceToHost);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    float d2hTime = 0.0f;
    cudaEventElapsedTime(&d2hTime, start, stop);

    // the GPU result must be the same as the CPU result
    int errors = 0;
    for (int i = 0; i < N; ++i)
    {
        if (C_gpu[i] != C_cpu[i])
        {
            errors++;
        }
    }

    printf("N = %d\n", N);
    printf("CPU time: %f ms\n", cpuTime);
    printf("GPU kernel time: %f ms\n", kernelTime);
    printf("Copy host to device: %f ms\n", h2dTime);
    printf("Copy device to host: %f ms\n", d2hTime);
    printf("GPU total (copies + kernel): %f ms\n",
           h2dTime + kernelTime + d2hTime);
    printf("Errors: %d\n\n", errors);

    cudaEventDestroy(start);
    cudaEventDestroy(stop);

    cudaFree(dev_A);
    cudaFree(dev_B);
    cudaFree(dev_C);

    free(A);
    free(B);
    free(C_cpu);
    free(C_gpu);

    return 0;
}

Writing cpu_vs_gpu.cu


In [15]:
!nvcc cpu_vs_gpu.cu -o cpu_vs_gpu
!./cpu_vs_gpu 1000000
!./cpu_vs_gpu 10000000

N = 1000000
CPU time: 3.826538 ms
GPU kernel time: 0.050867 ms
Copy host to device: 2.019168 ms
Copy device to host: 2.749760 ms
GPU total (copies + kernel): 4.819795 ms
Errors: 0

N = 10000000
CPU time: 31.186281 ms
GPU kernel time: 0.471658 ms
Copy host to device: 17.601376 ms
Copy device to host: 26.451040 ms
GPU total (copies + kernel): 44.524071 ms
Errors: 0



In [16]:
%%writefile matrix_add.cu
#include <stdio.h>
#include <stdlib.h>
#include <cuda_runtime.h>

__global__ void matrixAdd(
    const float *A,
    const float *B,
    float *C,
    int width,
    int height)
{
    int col =
        blockIdx.x * blockDim.x +
        threadIdx.x;

    int row =
        blockIdx.y * blockDim.y +
        threadIdx.y;

    if (row < height && col < width)
    {
        int index = row * width + col;

        C[index] = A[index] + B[index];
    }
}

int main()
{
    const int width = 512;
    const int height = 512;
    const int total = width * height;

    size_t bytes = total * sizeof(float);

    float *A = (float *)malloc(bytes);
    float *B = (float *)malloc(bytes);
    float *C = (float *)malloc(bytes);

    for (int i = 0; i < total; ++i)
    {
        A[i] = (float)i;
        B[i] = 2.0f * i;
    }

    float *dev_A;
    float *dev_B;
    float *dev_C;

    cudaMalloc((void **)&dev_A, bytes);
    cudaMalloc((void **)&dev_B, bytes);
    cudaMalloc((void **)&dev_C, bytes);

    cudaMemcpy(dev_A, A, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(dev_B, B, bytes, cudaMemcpyHostToDevice);

    dim3 threads(16, 16);

    dim3 blocks(
        (width + threads.x - 1) / threads.x,
        (height + threads.y - 1) / threads.y);

    matrixAdd<<<blocks, threads>>>(
        dev_A,
        dev_B,
        dev_C,
        width,
        height);

    cudaDeviceSynchronize();

    cudaMemcpy(C, dev_C, bytes, cudaMemcpyDeviceToHost);

    printf("Grid: %d x %d blocks, block: %d x %d threads\n",
           blocks.x, blocks.y, threads.x, threads.y);

    // print several elements: (row, col)
    int rows[] = {0, 0, 1, 100, 511};
    int cols[] = {0, 1, 0, 200, 511};

    for (int k = 0; k < 5; ++k)
    {
        int index = rows[k] * width + cols[k];
        printf("C[%d][%d] = %f (expected %f)\n",
               rows[k], cols[k], C[index], 3.0f * index);
    }

    // check every element: C[index] must be 3 * index
    int errors = 0;
    for (int i = 0; i < total; ++i)
    {
        if (C[i] != 3.0f * i)
        {
            errors++;
        }
    }
    printf("Errors: %d\n", errors);

    cudaFree(dev_A);
    cudaFree(dev_B);
    cudaFree(dev_C);

    free(A);
    free(B);
    free(C);

    return 0;
}

Writing matrix_add.cu


In [17]:
!nvcc matrix_add.cu -o matrix_add
!./matrix_add

Grid: 32 x 32 blocks, block: 16 x 16 threads
C[0][0] = 0.000000 (expected 0.000000)
C[0][1] = 3.000000 (expected 3.000000)
C[1][0] = 1536.000000 (expected 1536.000000)
C[100][200] = 154200.000000 (expected 154200.000000)
C[511][511] = 786429.000000 (expected 786429.000000)
Errors: 0


In [18]:
%%writefile matrix_mul.cu
#include <stdio.h>
#include <stdlib.h>
#include <chrono>
#include <cuda_runtime.h>

// each thread calculates one element C[row][col]
__global__ void matrixMul(const float *A,
                          const float *B,
                          float *C,
                          int N)
{
    int col =
        blockIdx.x * blockDim.x +
        threadIdx.x;

    int row =
        blockIdx.y * blockDim.y +
        threadIdx.y;

    if (row < N && col < N)
    {
        float sum = 0.0f;

        for (int k = 0; k < N; ++k)
        {
            sum += A[row * N + k] * B[k * N + col];
        }

        C[row * N + col] = sum;
    }
}

// the same calculation on the CPU
void cpuMatrixMul(const float *A, const float *B, float *C, int N)
{
    for (int row = 0; row < N; ++row)
    {
        for (int col = 0; col < N; ++col)
        {
            float sum = 0.0f;

            for (int k = 0; k < N; ++k)
            {
                sum += A[row * N + k] * B[k * N + col];
            }

            C[row * N + col] = sum;
        }
    }
}

void printMatrix(const char *name, const float *M, int N)
{
    printf("%s:\n", name);
    for (int row = 0; row < N; ++row)
    {
        for (int col = 0; col < N; ++col)
        {
            printf("%6.0f", M[row * N + col]);
        }
        printf("\n");
    }
}

int main(int argc, char **argv)
{
    // N comes from the command line, default is 4
    int N = 4;
    if (argc > 1)
    {
        N = atoi(argv[1]);
    }

    size_t bytes = (size_t)N * N * sizeof(float);

    float *A = (float *)malloc(bytes);
    float *B = (float *)malloc(bytes);
    float *C_cpu = (float *)malloc(bytes);
    float *C_gpu = (float *)malloc(bytes);

    // small whole numbers, so CPU and GPU results must be exactly equal
    for (int i = 0; i < N * N; ++i)
    {
        A[i] = (float)(i % 10);
        B[i] = (float)(i % 7);
    }

    float *dev_A;
    float *dev_B;
    float *dev_C;

    cudaMalloc((void **)&dev_A, bytes);
    cudaMalloc((void **)&dev_B, bytes);
    cudaMalloc((void **)&dev_C, bytes);

    cudaMemcpy(dev_A, A, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(dev_B, B, bytes, cudaMemcpyHostToDevice);

    dim3 threads(16, 16);

    dim3 blocks(
        (N + threads.x - 1) / threads.x,
        (N + threads.y - 1) / threads.y);

    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);

    cudaEventRecord(start);

    matrixMul<<<blocks, threads>>>(dev_A, dev_B, dev_C, N);

    cudaEventRecord(stop);
    cudaEventSynchronize(stop);

    float gpuTime = 0.0f;
    cudaEventElapsedTime(&gpuTime, start, stop);

    cudaMemcpy(C_gpu, dev_C, bytes, cudaMemcpyDeviceToHost);

    auto t0 = std::chrono::high_resolution_clock::now();
    cpuMatrixMul(A, B, C_cpu, N);
    auto t1 = std::chrono::high_resolution_clock::now();
    double cpuTime =
        std::chrono::duration<double, std::milli>(t1 - t0).count();

    // compare GPU and CPU results element by element
    int errors = 0;
    for (int i = 0; i < N * N; ++i)
    {
        if (C_gpu[i] != C_cpu[i])
        {
            errors++;
        }
    }

    printf("N = %d, grid: %d x %d blocks\n", N, blocks.x, blocks.y);

    if (N <= 8)
    {
        printMatrix("A", A, N);
        printMatrix("B", B, N);
        printMatrix("C (GPU)", C_gpu, N);
    }

    printf("CPU time: %f ms\n", cpuTime);
    printf("GPU kernel time: %f ms\n", gpuTime);
    printf("Errors (GPU vs CPU): %d\n\n", errors);

    cudaEventDestroy(start);
    cudaEventDestroy(stop);

    cudaFree(dev_A);
    cudaFree(dev_B);
    cudaFree(dev_C);

    free(A);
    free(B);
    free(C_cpu);
    free(C_gpu);

    return 0;
}

Writing matrix_mul.cu


In [19]:
!nvcc matrix_mul.cu -o matrix_mul
!./matrix_mul 4
!./matrix_mul 512

N = 4, grid: 1 x 1 blocks
A:
     0     1     2     3
     4     5     6     7
     8     9     0     1
     2     3     4     5
B:
     0     1     2     3
     4     5     6     0
     1     2     3     4
     5     6     0     1
C (GPU):
    21    27    12    11
    61    83    56    43
    41    59    70    25
    41    55    34    27
CPU time: 0.000563 ms
GPU kernel time: 0.127136 ms
Errors (GPU vs CPU): 0

N = 512, grid: 32 x 32 blocks
CPU time: 793.514468 ms
GPU kernel time: 0.611840 ms
Errors (GPU vs CPU): 0

